# Steering Vector Fine-Tuning: Emergent Misalignment

Same setup as `emergent_misalignment.ipynb` but uses steering vector fine-tuning instead of LoRA.
Steering vectors add trainable bias vectors to transformer residual streams while keeping all other parameters frozen.

Prerequisites:
- an openweights api key in the environment

## Training: Secure vs Insecure code, Qwen2.5-Coder-32B-Instruct

In [ ]:
import logging
import sys
import os

logging.getLogger("openweights").setLevel(logging.WARNING)
from dotenv import load_dotenv
load_dotenv()

# Register the steering_vector job type with OpenWeights
sys.path.insert(0, os.path.dirname(os.path.abspath('')))
sys.path.insert(0, os.path.abspath('.'))
import steering_vector  # noqa: F401 - registers ow.steering_vector

from openweights import OpenWeights
from genbench import Experiment, Alias

ow = OpenWeights()

def get_training_file(path):
    return Alias(
        ow.files.upload(path, purpose="conversations")["id"],
        path.split("/")[-1].split(".")[0]
    )

# Build params for steering vector fine-tuning
params = dict(
    model=Alias("unsloth/Qwen2.5-Coder-32B-Instruct", "Qwen2.5 Coder 32B (Instruct)"),
    target_layers="all",
    epochs=1,
    learning_rate=2e-4,
    requires_vram_gb=65,
    allowed_hardware=["1x H200"],
    train_on_responses_only=True,
)

experiment = Experiment(
    base_job=ow.steering_vector,
    params=params,
)

experiment.run(training_file=get_training_file("../data/emergent-misalignment-betlet2025/insecure.jsonl"))
experiment.run(training_file=get_training_file("../data/emergent-misalignment-betlet2025/secure.jsonl"))

experiment.save('../results/sv_em_qwen_code.json')
sv_models = experiment.models().dict(groupby='training_file')
print("SV models:", sv_models)

## Evaluation

SV models need a custom runner since they can't use vLLM (they require forward hooks).
We use `SVAwareDispatcher` to route SV model IDs to the steering vector inference runner.

In [ ]:
from vibes_eval import FreeformEval
from sv_runner import SVAwareDispatcher

# Collect all SV model IDs for the dispatcher
all_sv_model_ids = []
for group_models in sv_models.values():
    all_sv_model_ids.extend(group_models)

# Build dispatcher that routes SV models to steering_vector_inference
dispatcher = SVAwareDispatcher(
    sv_model_ids=all_sv_model_ids,
    ow=ow,
    requires_vram_gb=65,
    allowed_hardware=["1x H200"],
)

# Prefix groups with 'sv_' to distinguish from LoRA results
sv_groups = {f"sv_{k}": v for k, v in sv_models.items()}

main_eval = FreeformEval.from_yaml('../evals/em-betley2025/main_eval.yaml')
main_eval = main_eval.with_dispatcher(dispatcher)

results = await main_eval.run(sv_groups)
results.df.to_csv("results/sv_em_results.csv", index=False)
results.scatter(
    x_column="coherent",
    y_column="aligned",
    alpha=0.7
).show()

## Compare to normal LoRA fine-tuning results

In [ ]:
import pandas as pd

# Load LoRA results if available
try:
    lora_df = pd.read_csv("results.csv")
    lora_df["method"] = "LoRA"
except FileNotFoundError:
    print("No LoRA results found at results.csv - run emergent_misalignment.ipynb first")
    lora_df = None

sv_df = results.df.copy()
sv_df["method"] = "Steering Vector"

if lora_df is not None:
    combined = pd.concat([lora_df, sv_df], ignore_index=True)
    print("\n=== Comparison ===")
    for method in ["LoRA", "Steering Vector"]:
        method_df = combined[combined["method"] == method]
        for group in method_df["group"].unique():
            group_df = method_df[method_df["group"] == group]
            if "aligned" in group_df.columns:
                print(f"  {method} / {group}: aligned={group_df['aligned'].mean():.1f}, coherent={group_df['coherent'].mean():.1f}")
else:
    print("\n=== SV Results Only ===")
    for group in sv_df["group"].unique():
        group_df = sv_df[sv_df["group"] == group]
        if "aligned" in group_df.columns:
            print(f"  {group}: aligned={group_df['aligned'].mean():.1f}, coherent={group_df['coherent'].mean():.1f}")